# 04 BM25

## Objetivos

* Implementar BM25 y comparar su ranking con el de TF-IDF + coseno sobre la misma colección de documentos.
* Calcular el aporte de cada término en BM25, identificar qué controlan $k_1$ y $b$, y justificar un cambio de posición usando las estadísticas de los documentos.

**1. Carga el corpus de Project Gutenberg**

Utilice el corpus de Project Gutenberg y el preprocesamiento de la práctica anterior.

Recupere una tabla con los campos **Identificador de documento | Título | Texto**

Calcule la columna `tokens` para la tabla, con lista de tokens normalizados de cada documento.

Calcula DF e IDF.

In [ ]:
import os
import math
import string
from collections import Counter

import numpy as np
import pandas as pd
from IPython.display import display

# Ruta al corpus de Project Gutenberg en la carpeta epub
ruta_corpus = '../epub'
limite_libros = 100

# Obtener las carpetas numéricas de los libros y ordenarlas
carpetas = []
for nombre in os.listdir(ruta_corpus):
    if nombre.isdigit():
        carpetas.append(nombre)
carpetas.sort(key=int)

# Función para obtener el título desde la cabecera de Gutenberg
def obtener_titulo(texto):
    for linea in texto.split('\n'):
        if linea.startswith('Title:'):
            return linea.replace('Title:', '').strip()
    return 'Sin título'

ids_documentos = []
titulos = []
textos_originales = []

for carpeta in carpetas:
    ruta_archivo = os.path.join(ruta_corpus, carpeta, f'pg{carpeta}.txt')
    if os.path.exists(ruta_archivo):
        with open(ruta_archivo, 'r', encoding='utf-8', errors='ignore') as f:
            texto = f.read()

        ids_documentos.append(f'pg{carpeta}')
        titulos.append(obtener_titulo(texto))
        textos_originales.append(texto)

        if len(textos_originales) == limite_libros:
            break

N = len(textos_originales)
print(f"Se cargaron {N} documentos desde '{ruta_corpus}'.")

# Stopwords 
stopwords = {
    "the", "and", "to", "of", "a", "i", "in", "it",
    "that", "was", "he", "is", "for", "as", "on", "with"
}

def preprocesar(texto):
    # 1. Minúsculas
    texto_limpio = texto.lower()
    # 2. Quitar puntuación
    texto_limpio = texto_limpio.translate(str.maketrans('', '', string.punctuation))
    # 3. Dividir en lista de palabras (tokenizar)
    palabras = texto_limpio.split()
    # 4. Filtrar stopwords y números
    palabras_filtradas = [p for p in palabras if p not in stopwords and p.isalpha()]
    return palabras_filtradas

# Tabla con Identificador de documento | Título | Texto y columna tokens
tabla_corpus = pd.DataFrame({
    'Identificador de documento': ids_documentos,
    'Título': titulos,
    'Texto': textos_originales
})

lista_tokens = []
for texto in tabla_corpus['Texto']:
    lista_tokens.append(preprocesar(texto))

tabla_corpus['tokens'] = lista_tokens

pd.set_option('display.max_colwidth', 60)
print("\nTabla del corpus (con columna tokens):")
display(tabla_corpus)

# Calcular DF contando cada documento una sola vez por término
df_terminos = {}
for tokens_doc in tabla_corpus['tokens']:
    for termino in set(tokens_doc):
        if termino not in df_terminos:
            df_terminos[termino] = 0
        df_terminos[termino] += 1

vocabulario = sorted(df_terminos.keys())

# IDF(t) = ln(N / DF(t)) + 1
def calcular_idf(df, total_docs):
    return math.log(total_docs / df) + 1

idf_terminos = {}
for termino in vocabulario:
    idf_terminos[termino] = calcular_idf(df_terminos[termino], N)

tabla_df_idf = pd.DataFrame({
    'Término': vocabulario,
    'DF': [df_terminos[t] for t in vocabulario],
    'IDF': [round(idf_terminos[t], 4) for t in vocabulario]
})

print(f"\nTamaño del vocabulario: {len(vocabulario)} términos")
print("Muestra de DF e IDF (términos de ejemplo):")
terminos_muestra = ['zenda', 'flatland', 'harpoon', 'tarzan', 'whale', 'alice', 'constitution', 'sea', 'time', 'life']
display(tabla_df_idf[tabla_df_idf['Término'].isin(terminos_muestra)].sort_values('DF').reset_index(drop=True))

Se cargaron 100 documentos desde '../epub'.

Tabla del corpus (con columna tokens):


,Identificador de documento,Título,Texto,tokens
0,pg1,The Declaration of Independence of the United States of ...,The Project Gutenberg eBook of The Declaration of Indepe...,"[project, gutenberg, ebook, declaration, independence, u..."
1,pg2,The United States Bill of Rights,The Project Gutenberg eBook of The United States Bill of...,"[project, gutenberg, ebook, united, states, bill, rights..."
2,pg3,John F. Kennedy's Inaugural Address,The Project Gutenberg eBook of John F. Kennedy's Inaugur...,"[project, gutenberg, ebook, john, f, kennedys, inaugural..."
3,pg4,Lincoln's Gettysburg Address,The Project Gutenberg eBook of Lincoln's Gettysburg Addr...,"[project, gutenberg, ebook, lincolns, gettysburg, addres..."
4,pg5,The United States Constitution,The Project Gutenberg eBook of The United States Constit...,"[project, gutenberg, ebook, united, states, constitution..."
...,...,...,...,...
95,pg100,The Complete Works of William Shakespeare,The Project Gutenberg eBook of The Complete Works of Wil...,"[project, gutenberg, ebook, complete, works, william, sh..."
96,pg101,The Hacker Crackdown: Law and Disorder on the Electronic...,The Project Gutenberg eBook of The Hacker Crackdown: Law...,"[project, gutenberg, ebook, hacker, crackdown, law, diso..."
97,pg102,The Tragedy of Pudd'nhead Wilson,The Project Gutenberg eBook of The Tragedy of Pudd'nhead...,"[project, gutenberg, ebook, tragedy, puddnhead, wilson, ..."
98,pg103,Around the World in Eighty Days,The Project Gutenberg eBook of Around the World in Eight...,"[project, gutenberg, ebook, around, world, eighty, days,..."



Tamaño del vocabulario: 124188 términos
Muestra de DF e IDF (términos de ejemplo):


,Término,DF,IDF
0,zenda,1,5.6052
1,flatland,2,4.9120
2,harpoon,3,4.5066
3,tarzan,5,3.9957
4,whale,12,3.1203
5,alice,17,2.7720
6,constitution,39,1.9416
7,sea,69,1.3711
8,time,96,1.0408
9,life,100,1.0000


**2. Benchmarking**

Seleccione **ocho consultas en el idioma del corpus** y registra en una línea qué busca el usuario con cada una. Incluye términos muy frecuentes, términos poco frecuentes, y consultas de distintas longitudes. Usa $df$ para justificar las elecciones.

Para **TF-IDF + coseno** conserva la representación dispersa de la matriz.

Para **BM25** define $k_1=1.2$, $b=0.75$. Mantén intactas las estadísticas de la colección al cambiar de consulta.

Para cada consulta, ordena por puntuación descendente y desempata por `doc_id` ascendente (trata los IDs como cadenas en ambos métodos). Presenta hasta diez documentos con score positivo por método. Una consulta sin términos en el vocabulario produce dos listas vacías.

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1. Definición de las 8 consultas en el idioma del corpus
consultas = [
    {
        'consulta': 'zenda',
        'necesidad': 'Buscar la novela de aventuras sobre el prisionero de Zenda.',
        'justificacion_df': 'Consulta de 1 término muy raro (df=1: solo aparece en 1 documento).'
    },
    {
        'consulta': 'flatland dimensions',
        'necesidad': 'Encontrar la obra sobre un mundo geométrico de múltiples dimensiones.',
        'justificacion_df': 'Consulta corta (2 términos) que combina un término muy raro (flatland, df=2) con uno de frecuencia media-baja (dimensions, df=23).'
    },
    {
        'consulta': 'tarzan jungle apes',
        'necesidad': 'Historias de aventuras de Tarzán criado por los simios en la selva.',
        'justificacion_df': 'Consulta de 3 términos poco frecuentes (tarzan df=5, apes df=14, jungle df=19).'
    },
    {
        'consulta': 'alice rabbit queen king',
        'necesidad': 'Relatos de Alicia en el País de las Maravillas, el conejo blanco y los reyes.',
        'justificacion_df': 'Consulta de 4 términos que mezcla términos poco frecuentes (alice df=17, rabbit df=21) con frecuentes (queen df=50, king df=63).'
    },
    {
        'consulta': 'internet network email computer users',
        'necesidad': 'Guías introductorias sobre Internet, redes de computadoras y correo electrónico.',
        'justificacion_df': 'Consulta de 5 términos con términos específicos (internet df=11, users df=15) y términos muy frecuentes presentes en las cabeceras del corpus (df=100).'
    },
    {
        'consulta': 'constitution congress president rights government',
        'necesidad': 'Documentos históricos sobre la Constitución, el Congreso y el gobierno de Estados Unidos.',
        'justificacion_df': 'Consulta de 5 términos de frecuencia media y alta (congress df=34, constitution df=39, president df=43, rights df=61, government df=62).'
    },
    {
        'consulta': 'whale hunting ship captain harpoon sea',
        'necesidad': 'Novelas de aventuras sobre la caza de ballenas con arpón en altamar.',
        'justificacion_df': 'Consulta larga (6 términos) que combina términos muy raros (harpoon df=3, whale df=12) con términos frecuentes (captain df=48, ship df=51, hunting df=52, sea df=69).'
    },
    {
        'consulta': 'man life day time world great',
        'necesidad': 'Obras literarias o ensayos con reflexiones generales sobre el ser humano, la vida y el tiempo.',
        'justificacion_df': 'Consulta larga (6 términos) compuesta exclusivamente por términos muy frecuentes en la colección (df entre 85 y 100).'
    }
]

# Tabla con las 8 consultas y sus valores de DF
filas_consultas = []
for idx_q, c in enumerate(consultas, start=1):
    c['tokens'] = preprocesar(c['consulta'])

    partes_df = []
    for t in c['tokens']:
        df_val = df_terminos.get(t, 0)
        partes_df.append(f"{t} (df={df_val})")

    filas_consultas.append({
        'N°': idx_q,
        'Consulta': c['consulta'],
        'Longitud': len(c['tokens']),
        'DF por término': ', '.join(partes_df),
        'Qué busca el usuario': c['necesidad'],
        'Justificación con DF': c['justificacion_df']
    })

pd.set_option('display.max_colwidth', None)
print("Consultas seleccionadas y justificación mediante DF:")
display(pd.DataFrame(filas_consultas))

# 2. Representación dispersa para TF-IDF + coseno
def sin_cambios(tokens):
    return tokens

vectorizador_tfidf = TfidfVectorizer(
    analyzer=sin_cambios,
    use_idf=True,
    smooth_idf=False,
    sublinear_tf=False,
    norm=None
)

# Se conserva la matriz dispersa CSR
matriz_tfidf_dispersa = vectorizador_tfidf.fit_transform(tabla_corpus['tokens'])
print(f"Matriz TF-IDF dispersa: tipo={type(matriz_tfidf_dispersa).__name__}, forma={matriz_tfidf_dispersa.shape}, elementos no cero={matriz_tfidf_dispersa.nnz}")

# 3. Estadísticas de la colección para BM25
K1_BASE = 1.2
B_BASE = 0.75

longitudes_docs = []
conteos_docs = []
for tokens in tabla_corpus['tokens']:
    longitudes_docs.append(len(tokens))
    conteos_docs.append(Counter(tokens))

avgdl = sum(longitudes_docs) / N
print(f"Estadísticas fijas de BM25: N={N}, avgdl={avgdl:.2f} tokens, k1={K1_BASE}, b={B_BASE}")

# Funciones de búsqueda y ranking con desempate por doc_id ascendente
def buscar_tfidf_coseno(tokens_consulta, top_k=10):
    tokens_validos = [t for t in tokens_consulta if t in df_terminos]
    if len(tokens_validos) == 0:
        return []

    vector_q = vectorizador_tfidf.transform([tokens_consulta])
    scores = cosine_similarity(vector_q, matriz_tfidf_dispersa)[0]

    resultados = []
    for i in range(N):
        if scores[i] > 0:
            resultados.append({
                'doc_id': str(ids_documentos[i]),
                'titulo': titulos[i],
                'score': float(scores[i]),
                'indice_doc': i
            })

    resultados.sort(key=lambda r: (-r['score'], str(r['doc_id'])))
    return resultados[:top_k]

def aporte_termino_bm25(termino, indice_doc, k1=K1_BASE, b=B_BASE):
    if termino not in idf_terminos:
        return 0.0

    tf = conteos_docs[indice_doc][termino]
    if tf == 0:
        return 0.0

    dl = longitudes_docs[indice_doc]
    idf = idf_terminos[termino]
    numerador = tf * (k1 + 1.0)
    denominador = tf + k1 * (1.0 - b + b * (dl / avgdl))
    return idf * (numerador / denominador)

def buscar_bm25(tokens_consulta, k1=K1_BASE, b=B_BASE, top_k=10):
    tokens_validos = [t for t in tokens_consulta if t in df_terminos]
    if len(tokens_validos) == 0:
        return []

    resultados = []
    for i in range(N):
        score = 0.0
        for t in tokens_validos:
            score += aporte_termino_bm25(t, i, k1=k1, b=b)

        if score > 0:
            resultados.append({
                'doc_id': str(ids_documentos[i]),
                'titulo': titulos[i],
                'score': float(score),
                'indice_doc': i
            })

    resultados.sort(key=lambda r: (-r['score'], str(r['doc_id'])))
    return resultados[:top_k]

# Ejecución de las 8 consultas en ambos métodos
for c in consultas:
    c['top_tfidf'] = buscar_tfidf_coseno(c['tokens'], top_k=10)
    c['top_bm25'] = buscar_bm25(c['tokens'], k1=K1_BASE, b=B_BASE, top_k=10)

# Consulta sin términos en el vocabulario
consulta_vacia = preprocesar('quantumcomputing blockchainxyzw')
print(
    "\nVerificación de consulta sin términos en el vocabulario ('quantumcomputing blockchainxyzw'):",
    "TF-IDF ->", buscar_tfidf_coseno(consulta_vacia),
    "| BM25 ->", buscar_bm25(consulta_vacia)
)

Consultas seleccionadas y justificación mediante DF:


,N°,Consulta,Longitud,DF por término,Qué busca el usuario,Justificación con DF
0,1,zenda,1,zenda (df=1),Buscar la novela de aventuras sobre el prisionero de Zenda.,Consulta de 1 término muy raro (df=1: solo aparece en 1 documento).
1,2,flatland dimensions,2,"flatland (df=2), dimensions (df=23)",Encontrar la obra sobre un mundo geométrico de múltiples dimensiones.,"Consulta corta (2 términos) que combina un término muy raro (flatland, df=2) con uno de frecuencia media-baja (dimensions, df=23)."
2,3,tarzan jungle apes,3,"tarzan (df=5), jungle (df=19), apes (df=14)",Historias de aventuras de Tarzán criado por los simios en la selva.,"Consulta de 3 términos poco frecuentes (tarzan df=5, apes df=14, jungle df=19)."
3,4,alice rabbit queen king,4,"alice (df=17), rabbit (df=21), queen (df=50), king (df=63)","Relatos de Alicia en el País de las Maravillas, el conejo blanco y los reyes.","Consulta de 4 términos que mezcla términos poco frecuentes (alice df=17, rabbit df=21) con frecuentes (queen df=50, king df=63)."
4,5,internet network email computer users,5,"internet (df=11), network (df=100), email (df=100), computer (df=100), users (df=15)","Guías introductorias sobre Internet, redes de computadoras y correo electrónico.","Consulta de 5 términos con términos específicos (internet df=11, users df=15) y términos muy frecuentes presentes en las cabeceras del corpus (df=100)."
5,6,constitution congress president rights government,5,"constitution (df=39), congress (df=34), president (df=43), rights (df=61), government (df=62)","Documentos históricos sobre la Constitución, el Congreso y el gobierno de Estados Unidos.","Consulta de 5 términos de frecuencia media y alta (congress df=34, constitution df=39, president df=43, rights df=61, government df=62)."
6,7,whale hunting ship captain harpoon sea,6,"whale (df=12), hunting (df=52), ship (df=51), captain (df=48), harpoon (df=3), sea (df=69)",Novelas de aventuras sobre la caza de ballenas con arpón en altamar.,"Consulta larga (6 términos) que combina términos muy raros (harpoon df=3, whale df=12) con términos frecuentes (captain df=48, ship df=51, hunting df=52, sea df=69)."
7,8,man life day time world great,6,"man (df=85), life (df=100), day (df=88), time (df=96), world (df=100), great (df=91)","Obras literarias o ensayos con reflexiones generales sobre el ser humano, la vida y el tiempo.",Consulta larga (6 términos) compuesta exclusivamente por términos muy frecuentes en la colección (df entre 85 y 100).


Matriz TF-IDF dispersa: tipo=csr_matrix, forma=(100, 124188), elementos no cero=680211
Estadísticas fijas de BM25: N=100, avgdl=66669.09 tokens, k1=1.2, b=0.75

Verificación de consulta sin términos en el vocabulario ('quantumcomputing blockchainxyzw'): TF-IDF -> [] | BM25 -> []



**3. Compara los rankings**

Muestra por consulta una tabla como esta. Usa los IDs para comparar.

| Posición | ID y título TF-IDF | Coseno | ID y título BM25 | Score BM25 |
|---:|---|---:|---|---:|
| 1 | … | … | … | … |

Calcula cuántos IDs comparten los dos top 10 con $|T_{\mathrm{TFIDF}}\cap T_{\mathrm{BM25}}|$. Reporta también el tamaño real de cada lista y el primer documento de cada método. Dos listas pueden tener muchos IDs en común y ordenarlos de forma distinta.

Elige una consulta y dos documentos que cambien de posición o cuya distancia en el ranking cambie. Registra $|d|$, $f(t,d)$, $df(t)$ y el aporte BM25 de cada término de consulta para ambos documentos. Relaciona esas cifras con la saturación, la longitud o la rareza del término. Usa los scores de cada método para explicar su propio ranking.

In [ ]:
# 1. Tablas comparativas por consulta y cálculo de intersección
resumen_comparacion = []

for idx_q, c in enumerate(consultas, start=1):
    lista_tfidf = c['top_tfidf']
    lista_bm25 = c['top_bm25']
    max_filas = max(len(lista_tfidf), len(lista_bm25))

    ids_tfidf = set([r['doc_id'] for r in lista_tfidf])
    ids_bm25 = set([r['doc_id'] for r in lista_bm25])
    interseccion = ids_tfidf & ids_bm25

    primero_tfidf = f"{lista_tfidf[0]['doc_id']} ({lista_tfidf[0]['titulo']})" if lista_tfidf else 'Ninguno'
    primero_bm25 = f"{lista_bm25[0]['doc_id']} ({lista_bm25[0]['titulo']})" if lista_bm25 else 'Ninguno'

    resumen_comparacion.append({
        'N°': idx_q,
        'Consulta': c['consulta'],
        '|T_TFIDF|': len(lista_tfidf),
        '|T_BM25|': len(lista_bm25),
        '|T_TFIDF ∩ T_BM25|': len(interseccion),
        'Primer documento TF-IDF': primero_tfidf,
        'Primer documento BM25': primero_bm25
    })

    filas_tabla = []
    for pos in range(max_filas):
        if pos < len(lista_tfidf):
            id_tit_tfidf = f"{lista_tfidf[pos]['doc_id']} - {lista_tfidf[pos]['titulo']}"
            score_cos = round(lista_tfidf[pos]['score'], 6)
        else:
            id_tit_tfidf = '—'
            score_cos = None

        if pos < len(lista_bm25):
            id_tit_bm25 = f"{lista_bm25[pos]['doc_id']} - {lista_bm25[pos]['titulo']}"
            score_bm25 = round(lista_bm25[pos]['score'], 4)
        else:
            id_tit_bm25 = '—'
            score_bm25 = None

        filas_tabla.append({
            'Posición': pos + 1,
            'ID y título TF-IDF': id_tit_tfidf,
            'Coseno': score_cos,
            'ID y título BM25': id_tit_bm25,
            'Score BM25': score_bm25
        })

    print(f"\nConsulta {idx_q}: '{c['consulta']}'")
    print(f"  Tamaño lista TF-IDF: {len(lista_tfidf)} | Tamaño lista BM25: {len(lista_bm25)} | |T_TFIDF ∩ T_BM25| = {len(interseccion)}")
    print(f"  Top-1 TF-IDF: {primero_tfidf} | Top-1 BM25: {primero_bm25}")
    display(pd.DataFrame(filas_tabla))

print("\nResumen general de coincidencia en el Top-10 por consulta:")
display(pd.DataFrame(resumen_comparacion))

# 2. Análisis de cambio de posición entre dos documentos
def analizar_cambio_posicion(consulta_str, docs_comparar):
    tokens_q = preprocesar(consulta_str)
    ranking_tfidf = buscar_tfidf_coseno(tokens_q, top_k=N)
    ranking_bm25 = buscar_bm25(tokens_q, top_k=N)

    pos_tfidf = {}
    for i, r in enumerate(ranking_tfidf):
        pos_tfidf[r['doc_id']] = i + 1

    pos_bm25 = {}
    for i, r in enumerate(ranking_bm25):
        pos_bm25[r['doc_id']] = i + 1

    scores_cos = cosine_similarity(vectorizador_tfidf.transform([tokens_q]), matriz_tfidf_dispersa)[0]

    print(f"\n=== Análisis de cambio de posición para la consulta: '{consulta_str}' ===")
    print(f"Longitud media de la colección (avgdl): {avgdl:.2f} tokens")

    filas_resumen_docs = []
    for doc_id in docs_comparar:
        idx = ids_documentos.index(doc_id)
        score_bm = 0.0
        for t in tokens_q:
            score_bm += aporte_termino_bm25(t, idx)

        filas_resumen_docs.append({
            'Documento': f"{doc_id} ({titulos[idx]})",
            '|d| (longitud)': longitudes_docs[idx],
            '|d| / avgdl': round(longitudes_docs[idx] / avgdl, 4),
            'Posición TF-IDF': pos_tfidf.get(doc_id, '-'),
            'Score Coseno': round(float(scores_cos[idx]), 6),
            'Posición BM25': pos_bm25.get(doc_id, '-'),
            'Score BM25': round(score_bm, 4)
        })
    display(pd.DataFrame(filas_resumen_docs))

    filas_terminos = []
    for t in tokens_q:
        fila = {
            'Término (t)': t,
            'df(t)': df_terminos[t],
            'IDF(t)': round(idf_terminos[t], 4)
        }
        for doc_id in docs_comparar:
            idx = ids_documentos.index(doc_id)
            tf = conteos_docs[idx][t]
            aporte_bm = aporte_termino_bm25(t, idx)
            peso_tfidf = tf * idf_terminos[t]
            fila[f'f(t, {doc_id})'] = tf
            fila[f'Peso TF-IDF ({doc_id})'] = round(peso_tfidf, 4)
            fila[f'Aporte BM25 ({doc_id})'] = round(aporte_bm, 4)
        filas_terminos.append(fila)

    display(pd.DataFrame(filas_terminos))

analizar_cambio_posicion('alice rabbit queen king', ['pg11', 'pg12'])
analizar_cambio_posicion('constitution congress president rights government', ['pg5', 'pg18'])


Consulta 1: 'zenda'
  Tamaño lista TF-IDF: 1 | Tamaño lista BM25: 1 | |T_TFIDF ∩ T_BM25| = 1
  Top-1 TF-IDF: pg95 (The prisoner of Zenda) | Top-1 BM25: pg95 (The prisoner of Zenda)


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,pg95 - The prisoner of Zenda,0.130376,pg95 - The prisoner of Zenda,12.1895



Consulta 2: 'flatland dimensions'
  Tamaño lista TF-IDF: 10 | Tamaño lista BM25: 10 | |T_TFIDF ∩ T_BM25| = 9
  Top-1 TF-IDF: pg97 (Flatland: A Romance of Many Dimensions) | Top-1 BM25: pg97 (Flatland: A Romance of Many Dimensions)


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,pg97 - Flatland: A Romance of Many Dimensions,0.286971,pg97 - Flatland: A Romance of Many Dimensions,16.0918
1,2,pg35 - The Time Machine,0.011448,pg35 - The Time Machine,5.2035
2,3,"pg83 - From the Earth to the moon; and, round the moon",0.003558,"pg83 - From the Earth to the moon; and, round the moon",5.0873
3,4,pg41 - The Legend of Sleepy Hollow,0.001822,pg82 - Ivanhoe: A Romance,4.2068
4,5,pg49 - Surfing the Internet: An Introduction,0.001638,pg22 - Roget's Thesaurus,4.0115
5,6,pg82 - Ivanhoe: A Romance,0.000982,pg18 - The Federalist Papers,3.8323
6,7,pg96 - The Monster Men,0.000707,pg49 - Surfing the Internet: An Introduction,3.8248
7,8,pg18 - The Federalist Papers,0.000660,pg96 - The Monster Men,3.8037
8,9,"pg23 - Narrative of the Life of Frederick Douglass, an American Slave",0.000652,pg41 - The Legend of Sleepy Hollow,3.7730
9,10,pg64 - The gods of Mars,0.000624,pg64 - The gods of Mars,3.5234



Consulta 3: 'tarzan jungle apes'
  Tamaño lista TF-IDF: 10 | Tamaño lista BM25: 10 | |T_TFIDF ∩ T_BM25| = 9
  Top-1 TF-IDF: pg78 (Tarzan of the Apes) | Top-1 BM25: pg78 (Tarzan of the Apes)


,Posición,ID y título TF-IDF,Coseno,ID y título BM25,Score BM25
0,1,pg78 - Tarzan of the Apes,0.485384,pg78 - Tarzan of the Apes,21.0762
1,2,pg81 - The Return of Tarzan,0.479318,pg85 - The Beasts of Tarzan,21.0589
2,3,pg85 - The Beasts of Tarzan,0.410021,pg92 - Tarzan and the Jewels of Opar,21.0175
3,4,pg92 - Tarzan and the Jewels of Opar,0.335136,pg81 - The Return of Tarzan,21.0069
4,5,pg90 - The Son of Tarzan,0.094672,pg90 - The Son of Tarzan,20.8102
5,6,pg96 - The Monster Men,0.042785,pg68 - The warlord of Mars,9.7948
6,7,pg64 - The gods of Mars,0.010093,"pg72 - Thuvia, maid of Mars",6.2451
7,8,"pg72 - Thuvia, maid of Mars",0.009168,pg64 - The gods of Mars,6.2317
8,9,pg21 - Three hundred Aesop’s fables,0.006248,pg21 - Three hundred Aesop’s fables,5.9839
9,10,pg3 - John F. Kennedy's Inaugural Address,0.005114,pg96 - The Monster Men,5.8071


... (14 outputs left out)

,Término (t),df(t),IDF(t),"f(t, pg11)",Peso TF-IDF (pg11),Aporte BM25 (pg11),"f(t, pg12)",Peso TF-IDF (pg12),Aporte BM25 (pg12)
0,alice,17,2.7720,383,1061.6595,6.0895,440,1219.6610,6.0903
1,rabbit,21,2.5606,43,110.1079,5.5616,0,0.0000,0.0000
2,queen,50,1.6931,65,110.0546,3.6934,172,291.2213,3.7124
3,king,63,1.4620,60,87.7221,3.1870,62,90.6462,3.1867



=== Análisis de cambio de posición para la consulta: 'constitution congress president rights government' ===
Longitud media de la colección (avgdl): 66669.09 tokens


,Documento,|d| (longitud),|d| / avgdl,Posición TF-IDF,Score Coseno,Posición BM25,Score BM25
0,pg5 (The United States Constitution),5518,0.0828,1,0.172025,8,18.1307
1,pg18 (The Federalist Papers),132513,1.9876,2,0.146343,1,19.2235


,Término (t),df(t),IDF(t),"f(t, pg5)",Peso TF-IDF (pg5),Aporte BM25 (pg5),"f(t, pg18)",Peso TF-IDF (pg18),Aporte BM25 (pg18)
0,constitution,39,1.9416,19,36.8906,4.1890,473,918.3808,4.2528
1,congress,34,2.0788,29,60.2855,4.5151,121,251.5360,4.4958
2,president,43,1.8440,32,59.0070,4.0098,122,224.9643,3.9884
3,rights,61,1.4943,1,1.4943,2.3918,152,227.1330,3.2429
4,government,62,1.4780,5,7.3902,3.0251,846,1250.4183,3.2437


En la consulta `'alice rabbit queen king'`, con TF-IDF + coseno el documento `pg12` queda primero (`0.470373`) y `pg11` segundo (`0.460359`), pero con BM25 `pg11` sube al primer lugar (`18.5314`) y `pg12` baja al tercero (`12.9894`). Esto ocurre porque en TF-IDF el peso crece linealmente con la frecuencia $f(t, d)$, así que `pg12` gana por repetir más veces `alice` ($440$ vs $383$) y `queen` ($172$ vs $65$). En cambio, BM25 satura las repeticiones altas y le da mucho peso a la palabra poco frecuente `rabbit` ($df = 21$), que sí aparece $43$ veces en `pg11` (aportando `5.5616` puntos) pero $0$ veces en `pg12`, mientras que ambos libros tienen una longitud $|d|$ parecida y menor al promedio $\text{avgdl}$ (`18918` y `20669` frente a `66669.09`). Algo similar pasa en `'constitution congress president rights government'`, donde `pg5` gana en coseno por ser un documento muy corto ($|d| = 5518$), pero en BM25 `pg18` sube al puesto 1 porque, aunque es más largo ($|d| = 132513$), repite muchas más veces las cinco palabras de la consulta.

**4. Identifica el efecto de los parámetros**

Mantén fijos la colección y el preprocesamiento. Escoge una consulta con coincidencias en documentos de distintas longitudes y frecuencias.

| Ejecución | $k_1$ | $b$ | Qué examinar |
|---|---:|---:|---|
| Base | 1.2 | 0.75 | Posiciones y aportes de referencia |
| Sin ajuste por longitud | 1.2 | 0.00 | Cambios en documentos más cortos o más largos que la media |
| Mayor efecto de las repeticiones | 2.0 | 0.75 | Cambios en el aporte de términos con frecuencias distintas |

Muestra las posiciones de los mismos documentos en las tres configuraciones. Si no se mueven, compara sus aportes por término y explica por qué el cambio de puntuación no alteró su orden.

In [4]:
# Se escoge la consulta 'alice rabbit queen king', que presenta coincidencias en documentos
# cortos (pg11, pg12), cercanos a la media (pg45, pg77) y muy largos (pg100).
consulta_param = 'alice rabbit queen king'
tokens_param = preprocesar(consulta_param)

configuraciones = [
    ('Base', 1.2, 0.75),
    ('Sin ajuste por longitud', 1.2, 0.00),
    ('Mayor efecto de las repeticiones', 2.0, 0.75)
]

rankings_config = {}
posiciones_config = {}
scores_config = {}

for nombre_cfg, k1_val, b_val in configuraciones:
    ranking_completo = buscar_bm25(tokens_param, k1=k1_val, b=b_val, top_k=N)
    rankings_config[nombre_cfg] = ranking_completo[:10]

    posiciones_config[nombre_cfg] = {}
    scores_config[nombre_cfg] = {}
    for idx, r in enumerate(ranking_completo):
        posiciones_config[nombre_cfg][r['doc_id']] = idx + 1
        scores_config[nombre_cfg][r['doc_id']] = r['score']

# Seguimiento de los mismos documentos (Top-6 de la configuración Base) en las tres configuraciones
docs_seguimiento = []
for r in rankings_config['Base'][:6]:
    docs_seguimiento.append(r['doc_id'])

filas_posiciones = []
for doc_id in docs_seguimiento:
    idx = ids_documentos.index(doc_id)
    dl = longitudes_docs[idx]
    filas_posiciones.append({
        'Documento': f"{doc_id} - {titulos[idx]}",
        '|d|': dl,
        '|d| / avgdl': round(dl / avgdl, 2),
        'Pos. Base (1.2, 0.75)': posiciones_config['Base'][doc_id],
        'Score Base': round(scores_config['Base'][doc_id], 4),
        'Pos. Sin ajuste (1.2, 0.00)': posiciones_config['Sin ajuste por longitud'][doc_id],
        'Score (b=0)': round(scores_config['Sin ajuste por longitud'][doc_id], 4),
        'Pos. Mayor rep. (2.0, 0.75)': posiciones_config['Mayor efecto de las repeticiones'][doc_id],
        'Score (k1=2)': round(scores_config['Mayor efecto de las repeticiones'][doc_id], 4)
    })

print(f"Consulta analizada: '{consulta_param}' (avgdl = {avgdl:.2f} tokens)")
print("Posiciones y puntuaciones de los mismos documentos en las tres configuraciones:")
display(pd.DataFrame(filas_posiciones))

# Aportes por término para los documentos analizados en las 3 configuraciones
filas_aportes = []
for doc_id in docs_seguimiento[:5]:
    idx = ids_documentos.index(doc_id)
    dl = longitudes_docs[idx]
    for nombre_cfg, k1_val, b_val in configuraciones:
        fila = {
            'Documento': f"{doc_id} ({titulos[idx]})",
            '|d|/avgdl': round(dl / avgdl, 2),
            'Ejecución': f"{nombre_cfg} (k1={k1_val}, b={b_val})",
            'Posición': posiciones_config[nombre_cfg][doc_id],
            'Score Total': round(scores_config[nombre_cfg][doc_id], 4)
        }
        for t in tokens_param:
            tf = conteos_docs[idx][t]
            ap = aporte_termino_bm25(t, idx, k1=k1_val, b=b_val)
            fila[f"Aporte '{t}'"] = f"{ap:.4f} (f={tf})"
        filas_aportes.append(fila)

print("\nComparación detallada de los aportes por término en cada configuración:")
display(pd.DataFrame(filas_aportes))

Consulta analizada: 'alice rabbit queen king' (avgdl = 66669.09 tokens)
Posiciones y puntuaciones de los mismos documentos en las tres configuraciones:


,Documento,|d|,|d| / avgdl,"Pos. Base (1.2, 0.75)",Score Base,"Pos. Sin ajuste (1.2, 0.00)",Score (b=0),"Pos. Mayor rep. (2.0, 0.75)",Score (k1=2)
0,pg11 - Alice's Adventures in Wonderland,18918,0.28,1,18.5314,1,18.3706,1,25.1435
1,pg45 - Anne of Green Gables,71743,1.08,2,13.3335,3,13.5328,4,15.4460
2,pg12 - Through the Looking-Glass,20669,0.31,3,12.9894,4,12.9362,2,17.6677
3,pg100 - The Complete Works of William Shakespeare,720393,10.81,4,12.8006,2,16.8307,3,15.9335
4,pg77 - The House of the Seven Gables,73531,1.10,5,12.6114,5,12.8302,5,15.1475
5,pg54 - The Marvelous Land of Oz,29070,0.44,6,10.8837,9,10.0569,6,13.8034



Comparación detallada de los aportes por término en cada configuración:


,Documento,|d|/avgdl,Ejecución,Posición,Score Total,Aporte 'alice',Aporte 'rabbit',Aporte 'queen',Aporte 'king'
0,pg11 (Alice's Adventures in Wonderland),0.28,"Base (k1=1.2, b=0.75)",1,18.5314,6.0895 (f=383),5.5616 (f=43),3.6934 (f=65),3.1870 (f=60)
1,pg11 (Alice's Adventures in Wonderland),0.28,"Sin ajuste por longitud (k1=1.2, b=0.0)",1,18.3706,6.0793 (f=383),5.4805 (f=43),3.6574 (f=65),3.1534 (f=60)
2,pg11 (Alice's Adventures in Wonderland),0.28,"Mayor efecto de las repeticiones (k1=2.0, b=0.75)",1,25.1435,8.2958 (f=383),7.5201 (f=43),5.0081 (f=65),4.3195 (f=60)
3,pg45 (Anne of Green Gables),1.08,"Base (k1=1.2, b=0.75)",2,13.3335,4.2860 (f=3),3.4471 (f=2),3.3398 (f=11),2.2606 (f=3)
4,pg45 (Anne of Green Gables),1.08,"Sin ajuste por longitud (k1=1.2, b=0.0)",3,13.5328,4.3559 (f=3),3.5209 (f=2),3.3585 (f=11),2.2975 (f=3)
5,pg45 (Anne of Green Gables),1.08,"Mayor efecto de las repeticiones (k1=2.0, b=0.75)",4,15.4460,4.8781 (f=3),3.7344 (f=2),4.2606 (f=11),2.5729 (f=3)
6,pg12 (Through the Looking-Glass),0.31,"Base (k1=1.2, b=0.75)",3,12.9894,6.0903 (f=440),0.0000 (f=0),3.7124 (f=172),3.1867 (f=62)
7,pg12 (Through the Looking-Glass),0.31,"Sin ajuste por longitud (k1=1.2, b=0.0)",4,12.9362,6.0817 (f=440),0.0000 (f=0),3.6991 (f=172),3.1554 (f=62)
8,pg12 (Through the Looking-Glass),0.31,"Mayor efecto de las repeticiones (k1=2.0, b=0.75)",2,17.6677,8.2977 (f=440),0.0000 (f=0),5.0511 (f=172),4.3189 (f=62)
9,pg100 (The Complete Works of William Shakespeare),10.81,"Base (k1=1.2, b=0.75)",4,12.8006,4.6081 (f=31),1.2975 (f=3),3.6892 (f=1036),3.2058 (f=3004)


Al quitar el ajuste por longitud ($b = 0.00$), BM25 deja de penalizar a los documentos grandes, por lo que `pg100` (que mide más de 10 veces el promedio) sube del puesto 4 al 2 al aumentar el aporte de sus palabras poco repetidas como `rabbit` (`1.2975` a `4.0239`), mientras que documentos cortos como `pg12` pierden esa ventaja y bajan del puesto 3 al 4. Por otro lado, al aumentar el efecto de las repeticiones ($k_1 = 2.0$), se premia más a los libros donde las palabras aparecen muchas veces: `pg12` sube del puesto 3 al 2 porque repite bastante `alice` ($440$), `queen` ($172$) y `king` ($62$), mientras que `pg45` baja del puesto 2 al 4 porque tiene frecuencias muy bajas ($f = 3, 2, 11, 3$) y su puntaje crece poco. Finalmente, `pg11` no se mueve del puesto 1 en ninguna configuración porque es corto y tiene frecuencias altas en las cuatro palabras a la vez, por lo que aunque su puntaje total cambie (`18.5314`, `18.3706` y `25.1435`), la suma de sus aportes siempre supera a la de los demás documentos.